In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import cv2
import numpy as np
import os
from glob import glob

def analyze_image(path):
    img = cv2.imread(path)

    if img is None:
        return None

    h, w, c = img.shape
    aspect_ratio = w / h

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    mean = np.mean(gray)
    std = np.std(gray)
    blur = cv2.Laplacian(gray, cv2.CV_64F).var()

    _, thresh = cv2.threshold(gray, 127, 255, cv2.THRESH_BINARY_INV)
    fg_ratio = np.sum(thresh > 0) / (h * w)

    return {
        "height": h,
        "width": w,
        "aspect_ratio": aspect_ratio,
        "channels": c,
        "mean_intensity": mean,
        "std_intensity": std,
        "blur_score": blur,
        "foreground_ratio": fg_ratio
    }

In [3]:
def process_dataset(base_path):
    paths = glob(os.path.join(base_path, "**/*.png"), recursive=True)

    results = []

    for p in paths:
        res = analyze_image(p)
        if res is not None:
            results.append(res)

    return results

In [4]:
def compute_mean_stats(results):
    keys = results[0].keys()

    mean_stats = {}
    std_stats = {}

    for k in keys:
        values = [r[k] for r in results]

        mean_stats[k] = np.mean(values)
        std_stats[k] = np.std(values)

    return mean_stats, std_stats

In [5]:
base_path = "/kaggle/input/datasets/shreelakshmigp/cedardataset/signatures"

results = process_dataset(base_path)

mean_stats, std_stats = compute_mean_stats(results)

print("MEAN STATS:\n", mean_stats)
print("\nSTD STATS:\n", std_stats)

MEAN STATS:
 {'height': np.float64(350.4765151515152), 'width': np.float64(543.8397727272727), 'aspect_ratio': np.float64(1.6248611562063127), 'channels': np.float64(3.0), 'mean_intensity': np.float64(242.16433673433616), 'std_intensity': np.float64(20.341881897471335), 'blur_score': np.float64(392.8066543488104), 'foreground_ratio': np.float64(0.014144405335815543)}

STD STATS:
 {'height': np.float64(85.0988608333188), 'width': np.float64(96.8683135011362), 'aspect_ratio': np.float64(0.4587529199265432), 'channels': np.float64(0.0), 'mean_intensity': np.float64(7.068290088614342), 'std_intensity': np.float64(5.6128416977695945), 'blur_score': np.float64(211.70945155636244), 'foreground_ratio': np.float64(0.012258005844068592)}
